# Load Data

In [13]:
# Load data
import pandas as pd
import numpy as np

df = pd.read_csv("/content/train (1).csv")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


# Inspeksi Awal

In [14]:
for col in df.columns:
    dtype = df[col].dtype
    n_null = df[col].isnull().sum()
    n_unique = df[col].nunique()
    print(f"\n{'='*45}")
    print(f"KOLOM: {col} | tipe: {dtype} | null: {n_null} | unique: {n_unique}")

    # KOLOM ANGKA
    if dtype in ["float64", "int64"]:
        print(df[col].describe().to_string())
        # nilai negatif yang tidak wajar
        if (df[col] < 0).any():
            print(f"  ⚠️  Ada nilai negatif: {(df[col] < 0).sum()} baris")
        # nilai nol
        if (df[col] == 0).any():
            print(f"  ⚠️  Ada nilai nol: {(df[col] == 0).sum()} baris")
        # std sangat besar dibanding mean → indikasi outlier parah
        if df[col].std() > df[col].mean() * 3:
            print(f"  ⚠️  Std jauh lebih besar dari mean → distribusi mencurigakan")

    # KOLOM TEKS
    elif dtype == "object":
        print(f"  Contoh nilai: {df[col].unique()[:5]}")
        # cek apakah kolom angka tapi terbaca string
        sample = df[col].dropna().head(100)
        bisa_jadi_angka = sample.str.replace(",", "").str.replace(".", "").str.isnumeric().mean()
        if bisa_jadi_angka > 0.8:
            print(f"  ⚠️  Kemungkinan kolom angka terbaca sebagai string!")
        # cek inkonsistensi kapitalisasi
        if df[col].str.lower().nunique() < n_unique:
            print(f"  ⚠️  Ada inkonsistensi huruf besar/kecil")
        # cek whitespace tersembunyi
        if (df[col] != df[col].str.strip()).any():
            print(f"  ⚠️  Ada spasi tersembunyi di awal/akhir nilai")


KOLOM: Row ID | tipe: int64 | null: 0 | unique: 9800
count    9800.000000
mean     4900.500000
std      2829.160653
min         1.000000
25%      2450.750000
50%      4900.500000
75%      7350.250000
max      9800.000000

KOLOM: Order ID | tipe: object | null: 0 | unique: 4922
  Contoh nilai: ['CA-2017-152156' 'CA-2017-138688' 'US-2016-108966' 'CA-2015-115812'
 'CA-2018-114412']

KOLOM: Order Date | tipe: object | null: 0 | unique: 1230
  Contoh nilai: ['08/11/2017' '12/06/2017' '11/10/2016' '09/06/2015' '15/04/2018']

KOLOM: Ship Date | tipe: object | null: 0 | unique: 1326
  Contoh nilai: ['11/11/2017' '16/06/2017' '18/10/2016' '14/06/2015' '20/04/2018']

KOLOM: Ship Mode | tipe: object | null: 0 | unique: 4
  Contoh nilai: ['Second Class' 'Standard Class' 'First Class' 'Same Day']

KOLOM: Customer ID | tipe: object | null: 0 | unique: 793
  Contoh nilai: ['CG-12520' 'DV-13045' 'SO-20335' 'BH-11710' 'AA-10480']

KOLOM: Customer Name | tipe: object | null: 0 | unique: 793
  Contoh ni

# Deteksi Anomali

In [15]:
print(df[df["Sales"] > 5000][["Customer Name", "Product Name", "Category", "Sales"]].sort_values("Sales", ascending=False))

             Customer Name                                       Product Name  \
2697           Sean Miller  Cisco TelePresence System EX90 Videoconferenci...   
6826          Tamara Chand              Canon imageCLASS 2200 Advanced Copier   
8153          Raymond Buch              Canon imageCLASS 2200 Advanced Copier   
2623          Tom Ashbrook              Canon imageCLASS 2200 Advanced Copier   
4190          Hunter Lopez              Canon imageCLASS 2200 Advanced Copier   
9039         Adrian Barton   GBC Ibimaster 500 Manual ProClick Binding System   
4098          Sanjit Chand               Ibico EPK-21 Electric Binding System   
4277          Bill Shonely   3D Systems Cube Printer, 2nd Generation, Magenta   
8488          Sanjit Engle  HP Designjet T520 Inkjet Large Format Printer ...   
6425    Christopher Conant              Canon imageCLASS 2200 Advanced Copier   
2505          Ken Lonsdale        High Speed Automatic Electric Letter Opener   
165           Becky Martin  

In [16]:
print(df["Postal Code"].isnull().sum())
print(df[df["Postal Code"].isnull()][["City", "State", "Postal Code"]])

11
            City    State  Postal Code
2234  Burlington  Vermont          NaN
5274  Burlington  Vermont          NaN
8798  Burlington  Vermont          NaN
9146  Burlington  Vermont          NaN
9147  Burlington  Vermont          NaN
9148  Burlington  Vermont          NaN
9386  Burlington  Vermont          NaN
9387  Burlington  Vermont          NaN
9388  Burlington  Vermont          NaN
9389  Burlington  Vermont          NaN
9741  Burlington  Vermont          NaN


## Fix Postal Code

In [17]:
# Konversi dulu ke string, BARU isi nilai Burlington Vermont
df["Postal Code"] = df["Postal Code"].astype(str).str.replace(".0", "", regex=False).str.zfill(5)
df.loc[(df["City"] == "Burlington") & (df["State"] == "Vermont"), "Postal Code"] = "05401"

print(df[df["State"] == "Vermont"][["City", "State", "Postal Code"]])

            City    State Postal Code
2234  Burlington  Vermont       05401
5274  Burlington  Vermont       05401
8798  Burlington  Vermont       05401
9146  Burlington  Vermont       05401
9147  Burlington  Vermont       05401
9148  Burlington  Vermont       05401
9386  Burlington  Vermont       05401
9387  Burlington  Vermont       05401
9388  Burlington  Vermont       05401
9389  Burlington  Vermont       05401
9741  Burlington  Vermont       05401


In [18]:
print(df["Postal Code"].isnull().sum())

0


## Drop kolom Country karena semua isinya sama (tidak informatif)


In [19]:
df = df.drop(columns=["Country"])
print("Kolom tersisa:", df.columns.tolist())

Kolom tersisa: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales']


## Konversi tanggal + hitung Ship Duration

In [20]:
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True)
df["Ship Date"]  = pd.to_datetime(df["Ship Date"],  dayfirst=True)

df["Ship Duration"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Order Year"]    = df["Order Date"].dt.year
df["Order Month"]   = df["Order Date"].dt.month
df["Order Quarter"] = df["Order Date"].dt.quarter

print(df[["Order Date", "Ship Date", "Ship Duration", "Order Year", "Order Quarter"]].head())

  Order Date  Ship Date  Ship Duration  Order Year  Order Quarter
0 2017-11-08 2017-11-11              3        2017              4
1 2017-11-08 2017-11-11              3        2017              4
2 2017-06-12 2017-06-16              4        2017              2
3 2016-10-11 2016-10-18              7        2016              4
4 2016-10-11 2016-10-18              7        2016              4


In [21]:
# Validasi urutan tanggal — Ship Date tidak boleh sebelum Order Date
invalid = df[df["Ship Date"] < df["Order Date"]]
print("Tanggal tidak valid:", len(invalid), "baris")

Tanggal tidak valid: 0 baris


In [22]:
# Cleaning Final

In [23]:
# Simpan hasil cleaning final
df.to_csv("superstore_cleaned.csv", index=False)
print("✅ Data bersih tersimpan")
print(f"Shape final: {df.shape}")

✅ Data bersih tersimpan
Shape final: (9800, 21)
